# DS-3002 Data Mining — Assignment #3
## Pulse to Prediction: Mining Temporal Patterns and Building Classifiers on Real Patient Health Data
**Spring 2026 | BSDS | FAST-NUCES**

## Install Dependencies

In [ ]:
# ─── Install all required libraries ───────────────────────────────────────────
!pip install -q statsmodels dtaidistance imodels scikit-learn pandas numpy matplotlib seaborn

## Global Imports

In [ ]:
# ─── Core imports ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('All imports successful.')

---
# Required Preprocessing [15 Marks]

In [ ]:
# ─── P1: Load dataset and parse Timestamp ─────────────────────────────────────
# Upload patient_vitals.csv to Colab (Files panel) or mount Google Drive
# from google.colab import files; files.upload()

df = pd.read_csv('patient_vitals.csv', parse_dates=['Timestamp'])

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# ─── P2: Verify no null values in key columns ─────────────────────────────────
key_cols = ['PatientID', 'Timestamp', 'Diagnosis']
null_report = df[key_cols].isnull().sum()
print('Null counts in key columns:')
print(null_report)

# Overall null report
print('\nFull null report:')
print(df.isnull().sum())

In [ ]:
# ─── P3: Apply physiological range filters ────────────────────────────────────
valid_ranges = {
    'HeartRate':              (40,  180),
    'BloodPressureSystolic':  (80,  200),
    'BloodPressureDiastolic': (50,  130),
    'BloodOxygenLevel':       (85,  100),
    'BodyTemperature':        (35.5, 41.0),
    'RespiratoryRate':        (10,   40),
    'SleepHours':             (2,    12),
    'StressLevel':            (1,    10),
}

df_clean = df.copy()
rows_before = len(df_clean)

print(f'{'Column':<30} {'Rows Dropped':>12}')
print('-' * 45)
for col, (lo, hi) in valid_ranges.items():
    before = len(df_clean)
    df_clean = df_clean[(df_clean[col] >= lo) & (df_clean[col] <= hi)]
    dropped = before - len(df_clean)
    print(f'{col:<30} {dropped:>12}')

print(f'\nTotal rows before: {rows_before}')
print(f'Total rows after:  {len(df_clean)}')
print(f'Total rows dropped: {rows_before - len(df_clean)}')

In [ ]:
# ─── P4: Sort by PatientID then Timestamp ────────────────────────────────────
df_clean = df_clean.sort_values(['PatientID', 'Timestamp']).reset_index(drop=True)
print('Sorted. Sample:')
df_clean[['PatientID', 'Timestamp', 'HeartRate', 'Diagnosis']].head()

In [ ]:
# ─── P5-P6: Preprocessing Report ─────────────────────────────────────────────
print('='*55)
print('PREPROCESSING REPORT')
print('='*55)
print(f'Total rows retained     : {len(df_clean):,}')
print(f'Unique patients         : {df_clean["PatientID"].nunique()}')
print(f'Date range              : {df_clean["Timestamp"].min()} — {df_clean["Timestamp"].max()}')
avg_readings = df_clean.groupby('PatientID').size().mean()
print(f'Avg readings/patient    : {avg_readings:.1f}')

print('\nClass Distribution:')
class_dist = df_clean.groupby('PatientID')['Diagnosis'].first().value_counts()
class_pct  = class_dist / class_dist.sum() * 100
dist_df = pd.DataFrame({'Count': class_dist, 'Percent (%)': class_pct.round(2)})
print(dist_df)

In [ ]:
# ─── P6: Build patient-level feature matrix ──────────────────────────────────
numeric_vitals = [
    'HeartRate', 'BloodPressureSystolic', 'BloodPressureDiastolic',
    'BloodOxygenLevel', 'BodyTemperature', 'RespiratoryRate',
    'SleepHours', 'StressLevel'
]

def compute_slope(x):
    """Linear trend slope via least squares."""
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x))
    return np.polyfit(t, x, 1)[0]

agg_funcs = {'mean': 'mean', 'std': 'std', 'min': 'min', 'max': 'max'}

# Aggregate stats
feat_stats = df_clean.groupby('PatientID')[numeric_vitals].agg(agg_funcs)
feat_stats.columns = ['_'.join(c) for c in feat_stats.columns]

# Slope features
slope_df = df_clean.groupby('PatientID')[numeric_vitals].agg(compute_slope)
slope_df.columns = [f'{c}_slope' for c in slope_df.columns]

# Static features (Age, Gender)
static = df_clean.groupby('PatientID').agg(
    Age=('Age', 'first'),
    Gender=('Gender', 'first'),
    Diagnosis=('Diagnosis', 'first')
)
static['Gender_enc'] = (static['Gender'] == 'M').astype(int)

# Combine
patient_features = pd.concat([feat_stats, slope_df, static[['Age', 'Gender_enc', 'Diagnosis']]], axis=1)
patient_features = patient_features.dropna()

print(f'Patient feature matrix shape: {patient_features.shape}')
print(f'Feature columns ({len(patient_features.columns)-1}): {[c for c in patient_features.columns if c != "Diagnosis"][:5]} ...')
patient_features.head(3)

---
# Part A: Time Series Analysis & Trend Detection [25 Marks]

In [ ]:
# ─── A1: Select 3 patients ────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)

def pick_patient(diagnosis):
    ids = df_clean[df_clean['Diagnosis'] == diagnosis]['PatientID'].unique()
    return rng.choice(ids)

patient_healthy    = pick_patient('Healthy')
patient_hypertension = pick_patient('Hypertension')
patient_arrhythmia = pick_patient('Arrhythmia')

selected = {
    'Healthy':      patient_healthy,
    'Hypertension': patient_hypertension,
    'Arrhythmia':   patient_arrhythmia,
}
print('Selected patients:')
for cls, pid in selected.items():
    print(f'  {cls}: {pid}')

In [ ]:
# ─── A1: HeartRate time series plots + descriptive stats ─────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
colors = {'Healthy': '#2ecc71', 'Hypertension': '#e74c3c', 'Arrhythmia': '#9b59b6'}

print(f'{'Patient':<10} {'Class':<14} {'Mean':>7} {'Std':>7} {'Min':>7} {'Max':>7} {'CV (%)':>8}')
print('-'*58)

for ax, (cls, pid) in zip(axes, selected.items()):
    ts = df_clean[df_clean['PatientID'] == pid].set_index('Timestamp')['HeartRate']
    ax.plot(ts.index, ts.values, color=colors[cls], linewidth=1.2, alpha=0.85)
    ax.set_title(f'{pid} — {cls}', fontweight='bold')
    ax.set_ylabel('Heart Rate (bpm)')
    ax.set_xlabel('Timestamp')
    ax.grid(True, alpha=0.3)
    # Annotation
    ax.axhline(ts.mean(), color='black', linestyle='--', linewidth=0.8, label=f'Mean={ts.mean():.1f}')
    ax.legend(fontsize=8)
    # Stats
    cv = ts.std() / ts.mean() * 100
    print(f'{pid:<10} {cls:<14} {ts.mean():>7.2f} {ts.std():>7.2f} {ts.min():>7.2f} {ts.max():>7.2f} {cv:>8.2f}')

plt.suptitle('A1 — HeartRate Time Series (3 Selected Patients)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('A1_heartrate_timeseries.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── A2: BloodPressureSystolic — Rolling Means ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)

for ax, (cls, pid) in zip(axes, selected.items()):
    ts = df_clean[df_clean['PatientID'] == pid].set_index('Timestamp')['BloodPressureSystolic']
    ma7  = ts.rolling(window=7,  min_periods=1).mean()
    ma14 = ts.rolling(window=14, min_periods=1).mean()

    ax.plot(ts.index, ts.values,      color='lightgray',  linewidth=1,   alpha=0.7, label='Raw')
    ax.plot(ts.index, ma7.values,     color='steelblue',  linewidth=1.5, label='7-reading MA')
    ax.plot(ts.index, ma14.values,    color='tomato',     linewidth=1.5, label='14-reading MA')
    ax.set_title(f'{pid} — {cls} | BP Systolic with Rolling Means', fontweight='bold')
    ax.set_ylabel('BP Systolic (mmHg)')
    ax.set_xlabel('Timestamp')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('A2 — BP Systolic: Raw vs. 7-reading & 14-reading Rolling Mean', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('A2_rolling_means.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── A2: Additive Time Series Decomposition ───────────────────────────────────
# Find patient with most readings
reading_counts = df_clean.groupby('PatientID').size()
longest_patient = reading_counts.idxmax()
print(f'Patient with most readings: {longest_patient} ({reading_counts[longest_patient]} readings)')

ts_long = df_clean[df_clean['PatientID'] == longest_patient].set_index('Timestamp')['BloodPressureSystolic']
ts_long = ts_long.asfreq('6h', method='pad')  # ensure uniform freq

# period = 4 readings/day → weekly = 28, use 4 for daily seasonality
decomp = seasonal_decompose(ts_long, model='additive', period=4, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
decomp.observed.plot(ax=axes[0], title='Observed', color='steelblue')
decomp.trend.plot(ax=axes[1],    title='Trend',    color='tomato')
decomp.seasonal.plot(ax=axes[2], title='Seasonal', color='seagreen')
decomp.resid.plot(ax=axes[3],    title='Residual', color='gray')
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.suptitle(f'A2 — Additive Decomposition: {longest_patient} (BP Systolic)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('A2_decomposition.png', bbox_inches='tight')
plt.show()

trend_vals = decomp.trend.dropna()
trend_slope = np.polyfit(np.arange(len(trend_vals)), trend_vals.values, 1)[0]
direction = 'upward' if trend_slope > 0.01 else ('downward' if trend_slope < -0.01 else 'stable')
print(f'\nTrend slope  : {trend_slope:.4f} mmHg/reading')
print(f'Trend range  : {trend_vals.min():.2f} – {trend_vals.max():.2f} mmHg')
print(f'Trend direction: {direction}')

In [ ]:
c

In [ ]:
# ─── A3: Top 5 patients by anomaly count ─────────────────────────────────────
top5 = df_a3.groupby('PatientID').agg(
    anomaly_count=('anomaly', 'sum'),
    Diagnosis=('Diagnosis', 'first')
).sort_values('anomaly_count', ascending=False).head(5)

print('Top 5 patients with most anomalous HeartRate readings:')
print(top5)

# Plot top anomaly patient
top_pid = top5.index[0]
ts_top = df_a3[df_a3['PatientID'] == top_pid].sort_values('Timestamp')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts_top['Timestamp'], ts_top['HeartRate'], color='steelblue', linewidth=1, alpha=0.8, label='HeartRate')
ax.scatter(
    ts_top[ts_top['anomaly']]['Timestamp'],
    ts_top[ts_top['anomaly']]['HeartRate'],
    color='red', zorder=5, s=30, label='Anomaly'
)
mu  = ts_top['HR_mean'].iloc[0]
sig = ts_top['HR_std'].iloc[0]
ax.axhline(mu + 2*sig, color='orange', linestyle='--', linewidth=0.9, label='μ+2σ')
ax.axhline(mu - 2*sig, color='orange', linestyle='--', linewidth=0.9, label='μ-2σ')
ax.set_title(f'A3 — HeartRate Anomalies: {top_pid} ({top5.loc[top_pid, "Diagnosis"]})', fontweight='bold')
ax.set_xlabel('Timestamp'); ax.set_ylabel('Heart Rate (bpm)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('A3_anomaly_plot.png', bbox_inches='tight')
plt.show()

---
# Part B: Similarity Search & Patient Matching [20 Marks]

In [ ]:
# ─── B: Prepare normalised feature matrix ─────────────────────────────────────
feature_cols = [c for c in patient_features.columns if c != 'Diagnosis']
X_all   = patient_features[feature_cols].values
y_all   = patient_features['Diagnosis'].values
pids    = patient_features.index.tolist()

scaler_b = StandardScaler()
X_scaled = scaler_b.fit_transform(X_all)
print(f'Feature matrix: {X_scaled.shape}')

In [ ]:
# ─── B1: Euclidean & Manhattan Distance — Top-3 NN for 10 query patients ──────
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances

rng2 = np.random.default_rng(SEED)
query_idx = rng2.choice(len(pids), size=10, replace=False)
query_pids = [pids[i] for i in query_idx]

rows = []
for qi in query_idx:
    qvec = X_scaled[qi:qi+1]
    # Mask out the query itself
    mask = np.ones(len(pids), dtype=bool); mask[qi] = False
    X_rest = X_scaled[mask]
    pids_rest = [p for j, p in enumerate(pids) if j != qi]
    y_rest    = [y for j, y in enumerate(y_all)  if j != qi]

    for metric_name, dist_fn in [('Euclidean', euclidean_distances), ('Manhattan', manhattan_distances)]:
        dists = dist_fn(qvec, X_rest).flatten()
        nn_idx = np.argsort(dists)[:3]
        for rank, ni in enumerate(nn_idx, 1):
            rows.append({
                'Query_PID': pids[qi],
                'Query_Class': y_all[qi],
                'Metric': metric_name,
                f'NN{rank}_ID': pids_rest[ni],
                f'NN{rank}_Class': y_rest[ni],
                f'NN{rank}_Dist': round(dists[ni], 4)
            })

# Pivot to wide format
nn_df = pd.DataFrame(rows)
# Build compact table
table_rows = []
for qi in query_idx:
    qp = pids[qi]
    for metric in ['Euclidean', 'Manhattan']:
        sub = nn_df[(nn_df['Query_PID'] == qp) & (nn_df['Metric'] == metric)]
        row = {'Query PID': qp, 'Q.Class': y_all[qi], 'Metric': metric}
        for rank in [1, 2, 3]:
            rs = sub[sub.get(f'NN{rank}_ID', sub.columns[0]).name if False else [c for c in sub.columns if f'NN{rank}_ID' in c]]
            # Direct indexing
            r_sub = nn_df[(nn_df['Query_PID'] == qp) & (nn_df['Metric'] == metric)]
            try:
                row[f'NN{rank}'] = f"{r_sub.iloc[rank-1][f'NN{rank}_ID']} ({r_sub.iloc[rank-1][f'NN{rank}_Class']}, {r_sub.iloc[rank-1][f'NN{rank}_Dist']})"
            except:
                row[f'NN{rank}'] = 'N/A'
        table_rows.append(row)

result_table = pd.DataFrame(table_rows)
print('B1 — Top-3 Nearest Neighbours (Euclidean vs Manhattan):')
pd.set_option('display.max_colwidth', 35)
print(result_table.to_string(index=False))

In [ ]:
# ─── B2: DTW-Based Similarity ─────────────────────────────────────────────────
from dtaidistance import dtw

FIXED_LEN = 20

# Filter patients with ≥ 20 readings
valid_pids = reading_counts[reading_counts >= FIXED_LEN].index.tolist()
print(f'Patients with ≥{FIXED_LEN} readings: {len(valid_pids)}')

# Extract first 20 HeartRate readings per patient
sequences = {}
for pid in valid_pids:
    seq = df_clean[df_clean['PatientID'] == pid].sort_values('Timestamp')['HeartRate'].values[:FIXED_LEN]
    sequences[pid] = seq.astype(np.float64)

pid_list = list(sequences.keys())
seqs     = np.array([sequences[p] for p in pid_list])
diag_map = df_clean.groupby('PatientID')['Diagnosis'].first().to_dict()

# Compute pairwise DTW (may take a minute for 500 patients)
print('Computing pairwise DTW matrix (first 100 patients for speed)...')
N_DTW = min(100, len(pid_list))  # limit for speed in Colab
seqs_sub   = seqs[:N_DTW]
pids_sub   = pid_list[:N_DTW]

dtw_matrix = dtw.distance_matrix_fast(seqs_sub)
print(f'DTW matrix shape: {dtw_matrix.shape}')

# Pick 3 query patients: one per class
dtw_query_classes = ['Healthy', 'Hypertension', 'Arrhythmia']
print('\nB2 — DTW Top-3 Nearest Neighbours:')
print(f'{"Query (Class)":<20} {"Rank":<6} {"Neighbour":<10} {"Class":<15} {"DTW Dist":>10}')
print('-'*65)

same_class_dtw = 0; total_dtw = 0
for cls in dtw_query_classes:
    # Find first patient of this class in subset
    candidates = [i for i, p in enumerate(pids_sub) if diag_map.get(p) == cls]
    if not candidates:
        continue
    qi = candidates[0]
    row_dists = dtw_matrix[qi].copy()
    row_dists[qi] = np.inf
    nn_idx = np.argsort(row_dists)[:3]
    for rank, ni in enumerate(nn_idx, 1):
        nn_cls = diag_map.get(pids_sub[ni], 'Unknown')
        print(f'{pids_sub[qi]+" ("+cls+")":<20} {rank:<6} {pids_sub[ni]:<10} {nn_cls:<15} {row_dists[ni]:>10.4f}')
        if nn_cls == cls:
            same_class_dtw += 1
        total_dtw += 1

print(f'\nDTW same-class match rate: {same_class_dtw}/{total_dtw} ({same_class_dtw/total_dtw*100:.1f}%)')

In [ ]:
# ─── B3: Clinical Similarity — New Patient Admission ─────────────────────────
# New patient raw vector (using features that exist in our matrix)
# We'll map the given features to those in our feature matrix
# Available in assignment: HR_mean, HR_std, BPS_mean, BPS_std, BOL_mean, Sleep_mean

# Build a query row aligned to our feature columns
query_raw = pd.DataFrame([patient_features[feature_cols].mean()], columns=feature_cols)  # default baseline
# Override with given values
if 'HeartRate_mean'              in query_raw.columns: query_raw['HeartRate_mean']              = 98.0
if 'HeartRate_std'               in query_raw.columns: query_raw['HeartRate_std']               = 18.0
if 'BloodPressureSystolic_mean'  in query_raw.columns: query_raw['BloodPressureSystolic_mean']  = 155.0
if 'BloodPressureSystolic_std'   in query_raw.columns: query_raw['BloodPressureSystolic_std']   = 12.0
if 'BloodOxygenLevel_mean'       in query_raw.columns: query_raw['BloodOxygenLevel_mean']       = 94.0
if 'SleepHours_mean'             in query_raw.columns: query_raw['SleepHours_mean']             = 4.5

query_scaled = scaler_b.transform(query_raw.values)

dists_new = euclidean_distances(query_scaled, X_scaled).flatten()
top5_idx  = np.argsort(dists_new)[:5]

print('B3 — Top-5 Nearest Neighbours for new patient:')
top5_classes = []
for rank, ni in enumerate(top5_idx, 1):
    cls = y_all[ni]
    top5_classes.append(cls)
    print(f'  Rank {rank}: {pids[ni]} | Class: {cls} | Dist: {dists_new[ni]:.4f}')

from collections import Counter
vote = Counter(top5_classes)
predicted_class = vote.most_common(1)[0][0]
confidence = vote[predicted_class] / 5
print(f'\nPredicted Diagnosis: {predicted_class}')
print(f'Confidence (majority vote): {confidence*100:.0f}%')

---
# Part C: Supervised Classification [40 Marks]

In [ ]:
# ─── C: Train/Test Split (80/20 stratified) ───────────────────────────────────
X = patient_features[feature_cols].values
y = patient_features['Diagnosis'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler_c = StandardScaler()
X_train_sc = scaler_c.fit_transform(X_train)
X_test_sc  = scaler_c.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print('Train class distribution:', dict(zip(*np.unique(y_train, return_counts=True))))

# Helper to print and plot evaluation
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc  = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_te, y_pred, average='macro', zero_division=0)
    print(f'\n{"="*55}')
    print(f'{name} — Evaluation Report')
    print(f'{"="*55}')
    print(f'Accuracy : {acc:.4f}')
    print(f'Macro Precision: {prec:.4f}')
    print(f'Macro Recall   : {rec:.4f}')
    print(f'Macro F1       : {f1:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred, zero_division=0))
    cm = confusion_matrix(y_te, y_pred, labels=np.unique(y))
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay(cm, display_labels=np.unique(y)).plot(ax=ax, colorbar=False)
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{name.replace(" ","_")}_cm.png', bbox_inches='tight')
    plt.show()
    return {'Classifier': name, 'Accuracy': round(acc,4),
            'Macro Precision': round(prec,4), 'Macro Recall': round(rec,4), 'Macro F1': round(f1,4)}

results = []

In [ ]:
# ─── C1: Decision Tree ────────────────────────────────────────────────────────
depths = [2, 4, 6, 8, 10]
val_accs = []
for d in depths:
    dt = DecisionTreeClassifier(criterion='entropy', max_depth=d, random_state=SEED)
    dt.fit(X_train_sc, y_train)
    val_accs.append(accuracy_score(y_test, dt.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(depths, val_accs, marker='o', color='steelblue')
ax.set_xlabel('Max Depth'); ax.set_ylabel('Test Accuracy')
ax.set_title('C1 — Decision Tree: Accuracy vs. Max Depth', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('C1_depth_tuning.png', bbox_inches='tight'); plt.show()

best_depth = depths[np.argmax(val_accs)]
print(f'Optimal depth: {best_depth} (accuracy={max(val_accs):.4f})')

dt_best = DecisionTreeClassifier(criterion='entropy', max_depth=best_depth, random_state=SEED)
res_c1  = evaluate_model('Decision Tree', dt_best, X_train_sc, X_test_sc, y_train, y_test)
results.append(res_c1)

# Tree visualisation (depth 3)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_best, feature_names=feature_cols, class_names=np.unique(y),
          filled=True, max_depth=3, ax=ax, fontsize=7, impurity=True, proportion=False)
plt.title('C1 — Decision Tree (up to depth 3)', fontweight='bold')
plt.tight_layout(); plt.savefig('C1_tree_viz.png', bbox_inches='tight'); plt.show()

# Feature importance
imp = pd.Series(dt_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('\nTop 5 Feature Importances:')
print(imp.head(5))

In [ ]:
# ─── C2: Rule-Based Classification ───────────────────────────────────────────
# Extract if-then rules from C1 Decision Tree
from sklearn.tree import _tree

def extract_rules(tree, feature_names, class_names):
    tree_  = tree.tree_
    feat   = [feature_names[i] if i != _tree.TREE_UNDEFINED else 'undefined' for i in tree_.feature]
    rules  = []

    def recurse(node, depth, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name      = feat[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node],  depth+1, conditions + [f'{name} <= {threshold:.3f}'])
            recurse(tree_.children_right[node], depth+1, conditions + [f'{name} > {threshold:.3f}'])
        else:
            cls_idx   = np.argmax(tree_.value[node])
            cls_label = class_names[cls_idx]
            support   = tree_.n_node_samples[node]
            correct   = int(tree_.value[node][0][cls_idx])
            cov_pct   = round(support / tree_.n_node_samples[0] * 100, 2)
            acc_pct   = round(correct / support * 100, 2) if support > 0 else 0
            rules.append({
                'conditions': conditions,
                'class': cls_label,
                'support': support,
                'coverage_pct': cov_pct,
                'accuracy_pct': acc_pct
            })
    recurse(0, 0, [])
    return rules

rules = extract_rules(dt_best, feature_cols, dt_best.classes_)
rules_sorted = sorted(rules, key=lambda r: -r['support'])

print(f'Total rules extracted: {len(rules_sorted)}')
print('\nTop 10 Rules by Coverage:')
for i, r in enumerate(rules_sorted[:10], 1):
    conds = ' AND '.join(r['conditions'][:4])  # truncate for readability
    if len(r['conditions']) > 4:
        conds += ' ...'
    print(f'  Rule {i:2d}: IF {conds}')
    print(f'          THEN Diagnosis = {r["class"]} (coverage: {r["coverage_pct"]}%, accuracy: {r["accuracy_pct"]}%)')
    print()

# Evaluate rule-based on test set (same as DT since rules come from DT)
res_c2 = evaluate_model('Rule-Based (DT-extracted)', dt_best, X_train_sc, X_test_sc, y_train, y_test)
res_c2['Classifier'] = 'Rule-Based'
results.append(res_c2)

In [ ]:
# ─── C3: k-Nearest Neighbour ─────────────────────────────────────────────────
k_values = [1, 3, 5, 7, 9, 11, 15, 21]
train_accs_knn = []
test_accs_knn  = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_sc, y_train)
    train_accs_knn.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_accs_knn.append(accuracy_score(y_test,  knn.predict(X_test_sc)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, train_accs_knn, marker='s', label='Train', color='steelblue')
ax.plot(k_values, test_accs_knn,  marker='o', label='Test',  color='tomato')
ax.set_xlabel('k'); ax.set_ylabel('Accuracy')
ax.set_title('C3 — kNN: Accuracy vs. k', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('C3_knn_tuning.png', bbox_inches='tight'); plt.show()

best_k = k_values[np.argmax(test_accs_knn)]
print(f'Best k: {best_k} (test acc={max(test_accs_knn):.4f})')

# Euclidean
knn_euc = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean')
res_c3e = evaluate_model(f'kNN (k={best_k}, Euclidean)', knn_euc, X_train_sc, X_test_sc, y_train, y_test)

# Manhattan
knn_man = KNeighborsClassifier(n_neighbors=best_k, metric='manhattan')
knn_man.fit(X_train_sc, y_train)
man_acc = accuracy_score(y_test, knn_man.predict(X_test_sc))
print(f'Manhattan distance test accuracy (k={best_k}): {man_acc:.4f}')

res_c3e['Classifier'] = 'kNN'
results.append(res_c3e)

In [ ]:
# ─── C4: Naïve Bayes ─────────────────────────────────────────────────────────
# Gaussian assumption check
check_features = ['HeartRate_mean', 'BloodPressureSystolic_mean', 'SleepHours_mean']
diagnoses      = np.unique(y)

fig, axes = plt.subplots(len(check_features), len(diagnoses), figsize=(18, 9), sharey='row')
for row, feat in enumerate(check_features):
    fidx = feature_cols.index(feat)
    for col, diag in enumerate(diagnoses):
        mask = y_train == diag
        data = X_train[:, fidx][mask]
        axes[row, col].hist(data, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
        axes[row, col].set_title(f'{diag}', fontsize=8)
        if col == 0:
            axes[row, col].set_ylabel(feat.replace('_', ' '), fontsize=8)

plt.suptitle('C4 — Feature Distributions by Diagnosis Class (Gaussian Check)', fontweight='bold')
plt.tight_layout(); plt.savefig('C4_gaussian_check.png', bbox_inches='tight'); plt.show()

# Train NB
gnb = GaussianNB()
res_c4 = evaluate_model('Naïve Bayes', gnb, X_train_sc, X_test_sc, y_train, y_test)
results.append(res_c4)

# Correlation matrix
corr = pd.DataFrame(X_train_sc, columns=feature_cols).corr()
top_corr = corr.abs().unstack().sort_values(ascending=False)
top_corr = top_corr[top_corr < 1.0].drop_duplicates()
print('\nTop 5 most correlated feature pairs:')
print(top_corr.head(5))

# Correlation heatmap (top features)
top_feats = imp.head(10).index.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr.loc[top_feats, top_feats], annot=True, fmt='.2f', cmap='coolwarm', ax=ax, center=0)
ax.set_title('C4 — Feature Correlation Heatmap (Top-10 Features)', fontweight='bold')
plt.tight_layout(); plt.savefig('C4_correlation_heatmap.png', bbox_inches='tight'); plt.show()

In [ ]:
# ─── C5: Support Vector Machine ──────────────────────────────────────────────
C_values = [0.1, 1, 10, 100]
kernels  = ['rbf', 'poly']
best_config = {}

print('SVM Kernel Tuning via 5-Fold CV:')
print(f'{"Kernel":<10} {"C":<8} {"Mean CV Acc":>12}')
print('-'*33)
for kernel in kernels:
    best_score = -1
    for C in C_values:
        kw = {'kernel': kernel, 'C': C, 'random_state': SEED, 'decision_function_shape': 'ovr'}
        if kernel == 'poly':
            kw['degree'] = 3
        svm_cv = SVC(**kw)
        scores = cross_val_score(svm_cv, X_train_sc, y_train, cv=5, scoring='accuracy')
        mean_s = scores.mean()
        print(f'{kernel:<10} {C:<8} {mean_s:>12.4f}')
        if mean_s > best_score:
            best_score = mean_s
            best_config[kernel] = C

print(f'\nBest C for RBF    : {best_config["rbf"]}')
print(f'Best C for Poly   : {best_config["poly"]}')

# Train best model
best_kernel = 'rbf'  # typically better; compare scores above and override if needed
svm_best = SVC(kernel=best_kernel, C=best_config[best_kernel], random_state=SEED,
               decision_function_shape='ovr')
res_c5 = evaluate_model('SVM', svm_best, X_train_sc, X_test_sc, y_train, y_test)
results.append(res_c5)

In [ ]:
# ─── C5: Decision Boundary via PCA (2D) ──────────────────────────────────────
pca = PCA(n_components=2, random_state=SEED)
X_train_2d = pca.fit_transform(X_train_sc)
X_test_2d  = pca.transform(X_test_sc)

svm_2d = SVC(kernel=best_kernel, C=best_config[best_kernel], random_state=SEED,
             decision_function_shape='ovr')
svm_2d.fit(X_train_2d, y_train)

# Meshgrid
x_min, x_max = X_train_2d[:,0].min()-1, X_train_2d[:,0].max()+1
y_min, y_max = X_train_2d[:,1].min()-1, X_train_2d[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])

le = LabelEncoder().fit(y)
Z_enc = le.transform(Z).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx, yy, Z_enc, alpha=0.3, cmap='tab10')
scatter = ax.scatter(X_test_2d[:,0], X_test_2d[:,1],
                     c=le.transform(y_test), cmap='tab10', edgecolors='k', s=40, linewidths=0.5)
handles = [mpatches.Patch(color=plt.cm.tab10(le.transform([c])[0]/len(le.classes_)), label=c)
           for c in le.classes_]
ax.legend(handles=handles, loc='best', fontsize=8)
ax.set_title(f'C5 — SVM Decision Boundary (PCA 2D, kernel={best_kernel})', fontweight='bold')
ax.set_xlabel('PCA Component 1'); ax.set_ylabel('PCA Component 2')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('C5_decision_boundary.png', bbox_inches='tight'); plt.show()

In [ ]:
# ─── Classifier Comparison Summary Table ─────────────────────────────────────
summary_df = pd.DataFrame(results)
summary_df = summary_df.set_index('Classifier')
print('\n' + '='*65)
print('CLASSIFIER COMPARISON SUMMARY')
print('='*65)
print(summary_df.to_string())

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
summary_df[['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']].plot(
    kind='bar', ax=ax, colormap='Set2', width=0.7, edgecolor='white'
)
ax.set_title('Classifier Comparison — Performance Metrics', fontweight='bold')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=25)
ax.legend(loc='lower right')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('C_classifier_comparison.png', bbox_inches='tight'); plt.show()

---
## Final Recommendation

*(Fill this cell with your written recommendation after reviewing the summary table.)*

Based on the comparison, the recommended classifier for deployment in the hospital's patient monitoring system would be determined by balancing:
- **Accuracy & F1**: Which model scored highest?
- **Interpretability**: Decision Tree / Rule-Based are directly explainable to clinicians; SVM is a black box.
- **Computational cost**: kNN is expensive at inference time (O(n) per prediction); SVM and Decision Tree are fast.
- **Clinical risk**: False negatives (missed Arrhythmia / Hypertension) are more dangerous than false positives. Prefer high recall for critical classes.

**Recommendation**: The **Decision Tree** (or its Rule-Based equivalent) offers a strong balance of competitive accuracy, full interpretability, and low inference latency — critical in a real-time hospital monitoring environment where clinicians must understand and trust every prediction before acting.